In [ ]:
pip install qiskit qiskit-nature

In [ ]:
# ============================================
# PUBLICATION-QUALITY PIPELINE:
# QEMC (N=128) vs JW/BK+QEMC vs QAOA (N=10)
# ============================================

import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from dataclasses import dataclass, asdict
from typing import List, Tuple

from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector
from scipy.optimize import minimize

from qiskit_nature.second_q.operators import FermionicOp
from qiskit_nature.second_q.mappers import JordanWignerMapper, BravyiKitaevMapper

# -----------------------------
# Reproducibility
# -----------------------------
GLOBAL_SEED = 42
rng = np.random.default_rng(GLOBAL_SEED)

# -----------------------------
# Graph (±1 weights) + adjacency
# -----------------------------
def build_graph_pm1(N: int, p: float, seed: int):
    local_rng = np.random.default_rng(seed)
    edges = []
    weights = {}
    adjacency = [[] for _ in range(N)]
    for i in range(N):
        for j in range(i+1, N):
            if local_rng.random() < p:
                w = local_rng.choice([1, -1])
                edges.append((i, j))
                weights[(i, j)] = w
                adjacency[i].append((j, w))
                adjacency[j].append((i, w))
    return edges, weights, adjacency

# -----------------------------
# QEMC ansatz (low-depth)
# -----------------------------
def qemc_ansatz(params: np.ndarray, n_qubits: int):
    qc = QuantumCircuit(n_qubits)
    qc.h(range(n_qubits))
    # one layer (reps=1)
    for i, p in enumerate(params):
        qc.ry(p, i % n_qubits)
    for i in range(n_qubits - 1):
        qc.cx(i, i+1)
    return qc

# -----------------------------
# Fast sampling cost (QEMC)
# -----------------------------
def qemc_cost(params, adjacency, N, n_qubits, shots=300):
    qc = qemc_ansatz(params, n_qubits)
    state = Statevector.from_instruction(qc)
    counts = state.sample_counts(shots)
    total = 0.0
    for b, c in counts.items():
        p = c / shots
        node = int(b[::-1], 2) % N
        for nb, w in adjacency[node]:
            total += p * w
    return -total  # maximize cut

# -----------------------------
# Fermionic -> JW/BK mapping
# (toy diagonal model to keep cost cheap)
# -----------------------------
def build_fermionic(N):
    terms = {}
    for i in range(N):
        terms[f"+_{i} -_{i}"] = 1.0
    return FermionicOp(terms, num_spin_orbitals=N)

def map_hamiltonian(N):
    ferm = build_fermionic(N)
    jw = JordanWignerMapper().map(ferm)
    bk = BravyiKitaevMapper().map(ferm)
    return jw, bk

# -----------------------------
# Hybrid cost (JW/BK + QEMC)
# -----------------------------
def hybrid_cost(params, qubit_op, N, n_qubits, shots=300):
    qc = qemc_ansatz(params, n_qubits)
    state = Statevector.from_instruction(qc)
    counts = state.sample_counts(shots)
    total = 0.0
    for b, c in counts.items():
        p = c / shots
        idx = int(b[::-1], 2) % N
        # simple Z-only readout from mapped operator
        for pauli, coeff in zip(qubit_op.paulis, qubit_op.coeffs):
            label = pauli.to_label()
            if idx < len(label) and label[idx] == "Z":
                total += p * coeff.real
    return total

# -----------------------------
# QAOA baseline (N small only)
# -----------------------------
def qaoa_ansatz(params, N):
    qc = QuantumCircuit(N)
    qc.h(range(N))
    for i in range(N):
        qc.ry(params[i], i)
    for i in range(N-1):
        qc.cx(i, i+1)
    return qc

def qaoa_cost(params, adjacency, N):
    qc = qaoa_ansatz(params, N)
    state = Statevector.from_instruction(qc)
    probs = state.probabilities_dict()
    total = 0.0
    for b, p in probs.items():
        bits = [int(x) for x in b[::-1]]
        for i in range(N):
            for j, w in adjacency[i]:
                if i < j and bits[i] != bits[j]:
                    total += p * w
    return -total

# -----------------------------
# Experiment runner
# -----------------------------
@dataclass
class RunResult:
    method: str
    N: int
    qubits: int
    best_value: float
    runtime_s: float
    seed: int

def optimize(fun, x0, args, maxiter=120):
    t0 = time.time()
    res = minimize(fun, x0, args=args, method='COBYLA',
                   options={'maxiter': maxiter, 'rhobeg': 0.5})
    return res, time.time() - t0

def run_trial(seed: int):
    results = []

    # ---- QEMC (N=128) ----
    N_big = 128
    nq = int(np.ceil(np.log2(N_big)))
    edges, weights, adjacency = build_graph_pm1(N_big, p=0.05, seed=seed)
    x0 = np.random.default_rng(seed).uniform(0, 2*np.pi, nq)

    res, dt = optimize(qemc_cost, x0, (adjacency, N_big, nq, 300), maxiter=150)
    results.append(RunResult("QEMC", N_big, nq, -res.fun, dt, seed))

    # ---- JW/BK + QEMC (N=128) ----
    jw_op, bk_op = map_hamiltonian(N_big)

    x0 = np.random.default_rng(seed+1).uniform(0, 2*np.pi, nq)
    res_jw, dt_jw = optimize(hybrid_cost, x0, (jw_op, N_big, nq, 300), maxiter=120)
    results.append(RunResult("JW+QEMC", N_big, nq, res_jw.fun, dt_jw, seed))

    x0 = np.random.default_rng(seed+2).uniform(0, 2*np.pi, nq)
    res_bk, dt_bk = optimize(hybrid_cost, x0, (bk_op, N_big, nq, 300), maxiter=120)
    results.append(RunResult("BK+QEMC", N_big, nq, res_bk.fun, dt_bk, seed))

    # ---- QAOA baseline (N=10) ----
    N_small = 10
    nq_small = N_small
    _, _, adj_small = build_graph_pm1(N_small, p=0.4, seed=seed+3)
    x0 = np.random.default_rng(seed+4).uniform(0, 2*np.pi, nq_small)
    res_qaoa, dt_qaoa = optimize(qaoa_cost, x0, (adj_small, N_small), maxiter=80)
    results.append(RunResult("QAOA", N_small, nq_small, -res_qaoa.fun, dt_qaoa, seed))

    return results

# -----------------------------
# Multi-run experiment
# -----------------------------
def run_experiment(num_trials=5):
    all_rows = []
    seeds = [GLOBAL_SEED + i for i in range(num_trials)]
    for s in seeds:
        rows = run_trial(s)
        all_rows.extend([asdict(r) for r in rows])
    df = pd.DataFrame(all_rows)
    return df

df = run_experiment(num_trials=5)

# -----------------------------
# Aggregate (mean ± std)
# -----------------------------
summary = (df
           .groupby(["method", "N", "qubits"])
           .agg(mean_value=("best_value", "mean"),
                std_value=("best_value", "std"),
                mean_time=("runtime_s", "mean"),
                std_time=("runtime_s", "std"),
                runs=("best_value", "count"))
           .reset_index())

print("\n=== SUMMARY TABLE ===")
print(summary.to_string(index=False))

# -----------------------------
# PLOTS (publication style)
# -----------------------------
plt.rcParams.update({
    "figure.figsize": (7,5),
    "axes.grid": True,
    "grid.linestyle": "--",
    "grid.alpha": 0.6,
    "font.size": 11
})

# 1) Performance (mean ± std)
plt.figure()
for m in summary["method"].unique():
    sub = summary[summary["method"] == m]
    # one point per method (N differs across methods)
    plt.errorbar(sub["N"], sub["mean_value"], yerr=sub["std_value"],
                 marker='o', capsize=4, label=m)
plt.xlabel("Problem size (N)")
plt.ylabel("Objective (MaxCut proxy / energy)")
plt.title("Performance (mean ± std)")
plt.legend()
plt.tight_layout()

# 2) Qubits used
plt.figure()
for m in summary["method"].unique():
    sub = summary[summary["method"] == m]
    plt.plot(sub["N"], sub["qubits"], marker='o', label=m)
plt.xlabel("Problem size (N)")
plt.ylabel("Qubits")
plt.title("Qubit scaling")
plt.legend()
plt.tight_layout()

# 3) Runtime
plt.figure()
for m in summary["method"].unique():
    sub = summary[summary["method"] == m]
    plt.errorbar(sub["N"], sub["mean_time"], yerr=sub["std_time"],
                 marker='o', capsize=4, label=m)
plt.xlabel("Problem size (N)")
plt.ylabel("Runtime (s)")
plt.title("Runtime (mean ± std)")
plt.legend()
plt.tight_layout()

# 4) Convergence example (single run for QEMC N=128)
# (optional: quick trace)
def convergence_trace(N=128, seed=GLOBAL_SEED):
    nq = int(np.ceil(np.log2(N)))
    _, _, adjacency = build_graph_pm1(N, p=0.05, seed=seed)
    history = []
    def cb(xk):
        history.append(qemc_cost(xk, adjacency, N, nq, shots=200))
    x0 = np.random.default_rng(seed).uniform(0, 2*np.pi, nq)
    minimize(qemc_cost, x0, args=(adjacency, N, nq, 200),
             method='COBYLA', options={'maxiter': 120}, callback=cb)
    return -np.array(history)

trace = convergence_trace()
plt.figure()
plt.plot(trace)
plt.xlabel("Iteration")
plt.ylabel("MaxCut value (proxy)")
plt.title("Convergence (QEMC, N=128)")
plt.tight_layout()

plt.show()

# -----------------------------
# Save artifacts
# -----------------------------
summary.to_csv("summary_table.csv", index=False)
df.to_csv("all_runs.csv", index=False)
# Figures can be saved similarly with plt.savefig(...) before show()